# Transfer Learning USAD (SWaT 51 sensores → SIATA 1 canal)

**Plan D** — Detectar anomalías de temperatura en la estación SIATA 68 reutilizando un USAD pre-entrenado sobre SWaT, vía **inicialización por submatriz** de un solo sensor donante.

Este notebook se ejecuta en **Google Colab** y clona el repo desde GitHub. No requiere Drive; los datos y el checkpoint viven en el propio repo.

## 1. Montaje Colab ↔ GitHub

In [ ]:
import os, sys

GITHUB_USER = "ronvas234"              # <-- Cambiar por tu usuario GitHub
REPO_NAME   = "data-science-monograph" # <-- Cambiar por el nombre del repo
BRANCH      = "feature/transfer-learning-plan-c"

REPO_DIR = f"/content/{REPO_NAME}"
USAD_DIR = f"{REPO_DIR}/modelos/usad"
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone -b {BRANCH} https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    !pip install -q torch pandas scikit-learn matplotlib seaborn
    os.chdir(USAD_DIR)
# En local el notebook vive en modelos/usad; si el kernel arranco en otro cwd, ajustalo manualmente.

sys.path.insert(0, os.getcwd())
print('cwd =', os.getcwd())

## 2. Imports y configuración

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# cwd está en modelos/usad/, así que importamos el paquete directo
from plan_d import (
    PathsConfig, DataConfig, ModelConfig, TrainingConfig,
    SiataCsvLoader, ChronologicalSplitter,
    MinMaxScalerPersistable, SlidingWindowizer, TemperatureWindowDataset,
    SubmatrixExtractor, SingleChannelUSAD,
    TransferLearningTrainer, FreezeInner, FullFinetune,
    ROCEvaluator, ClassificationEvaluator, YoudenJSelector, F1OptimalSelector,
    AnomalyDetector,
)
from plan_d.visualization import (
    plot_roc, plot_loss_history, plot_score_histogram, plot_anomaly_timeline,
)

# repo_root = dos niveles por encima de modelos/usad/
paths = PathsConfig(repo_root=Path.cwd().parent.parent)
data_cfg  = DataConfig()
model_cfg = ModelConfig()
train_cfg = TrainingConfig(epochs=30, batch_size=256, freeze_inner_layers=True)

torch.manual_seed(train_cfg.seed)
np.random.seed(train_cfg.seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
print('checkpoint existe:', paths.pretrained_checkpoint.exists())
print('csv existe       :', paths.data_csv.exists())

## 3. Carga y re-split del dataset

Se ignora la columna `Split` original y se rederivan train/val/test según el criterio del Plan D:
- **Train** = tramo anterior al gap de 41 días.
- **Val** = primer 30 % del tramo posterior al gap.
- **Test** = 70 % restante del tramo posterior.

In [ ]:
df = SiataCsvLoader(data_cfg).load(paths.data_csv)
print('df shape:', df.shape)
print('rango temporal:', df[data_cfg.timestamp_column].min(), '→', df[data_cfg.timestamp_column].max())

splits = ChronologicalSplitter(data_cfg).split(df)
splits.summary()

## 4. Escalado y ventanas deslizantes

`MinMaxScaler` se ajusta **solo** sobre train (evita data leakage) y se persiste con joblib. Las ventanas son de 12 timesteps (obligatorio por la arquitectura pre-entrenada).

In [ ]:
scaler = MinMaxScalerPersistable().fit(splits.train[data_cfg.value_column].values)
scaler.save(paths.scaler_path)
windowizer = SlidingWindowizer(data_cfg.window_size, data_cfg.stride)

def build_dataset(df_split):
    x = scaler.transform(df_split[data_cfg.value_column].values)
    w = windowizer.transform(x)
    y = windowizer.window_labels(df_split[data_cfg.anomaly_flag_column].values)
    return TemperatureWindowDataset(w, y)

ds_train = build_dataset(splits.train)
ds_val   = build_dataset(splits.val)
ds_test  = build_dataset(splits.test)

print('ventanas tr/va/te:', len(ds_train), len(ds_val), len(ds_test))
print('anomalías tr/va/te:', int(ds_train.labels.sum()), int(ds_val.labels.sum()), int(ds_test.labels.sum()))

loader_train = DataLoader(ds_train, batch_size=train_cfg.batch_size, shuffle=True,  drop_last=False)
loader_val   = DataLoader(ds_val,   batch_size=train_cfg.batch_size, shuffle=False, drop_last=False)
loader_test  = DataLoader(ds_test,  batch_size=train_cfg.batch_size, shuffle=False, drop_last=False)

## 5. Transfer learning por submatriz

Se extrae la rebanada del sensor donante `k` del `model.pth` pre-entrenado:
- `encoder.linear1.weight`: `(306, 612)` → reshape `(306, 12, 51)` → `[:, :, k]` → `(306, 12)`
- `decoder*.linear3.weight`: `(612, 306)` → reshape `(12, 51, 306)` → `[:, k, :]` → `(12, 306)`
- Las capas internas se copian tal cual.

In [ ]:
submatrix = SubmatrixExtractor(model_cfg).extract(paths.pretrained_checkpoint)
print('sensor donante elegido:', submatrix.chosen_sensor)
print('encoder.linear1.weight submatriz:', submatrix.encoder_state['linear1.weight'].shape)
print('decoder1.linear3.weight submatriz:', submatrix.decoder1_state['linear3.weight'].shape)

model = SingleChannelUSAD.from_submatrix(
    submatrix,
    window_size=model_cfg.pretrained_window_size,
    latent_size=model_cfg.latent_size,
).to(device)
print('parámetros totales:', sum(p.numel() for p in model.parameters()))

## 6. Entrenamiento (fine-tuning adversarial USAD)

Estrategia por defecto: `FreezeInner` — se congelan las capas internas (ya aprendidas sobre SWaT) y solo se entrenan las capas borde rehechas por submatriz. Cambie a `FullFinetune()` si el loss se estanca.

In [ ]:
strategy = FreezeInner() if train_cfg.freeze_inner_layers else FullFinetune()
trainer = TransferLearningTrainer(train_cfg, strategy, device)

history = trainer.fit(
    model, loader_train, loader_val,
    save_best_to=paths.finetuned_checkpoint,
)

fig_loss = plot_loss_history(history)
fig_loss.show()

## 7. Curva ROC en Train + Validation

La ROC de train es diagnóstica (debería dar AUC alto: el modelo vio esos datos). La ROC de val es el estándar para **seleccionar el umbral** óptimo.

In [ ]:
roc_eval = ROCEvaluator(device, alpha=train_cfg.alpha, beta=train_cfg.beta)

# Para train usamos orden no-aleatorio para que los labels coincidan
loader_train_eval = DataLoader(ds_train, batch_size=train_cfg.batch_size, shuffle=False)

scores_train = roc_eval.scores(model, loader_train_eval)
scores_val   = roc_eval.scores(model, loader_val)

roc_train = roc_eval.roc(scores_train, ds_train.labels)
roc_val   = roc_eval.roc(scores_val,   ds_val.labels)

print(f'AUC train = {roc_train.auc:.4f}')
print(f'AUC val   = {roc_val.auc:.4f}')
plot_roc(roc_train, roc_val).show()

## 8. Selección de umbral sobre validation

In [ ]:
youden_thr = YoudenJSelector().select(scores_val, ds_val.labels)
f1_thr     = F1OptimalSelector().select(scores_val, ds_val.labels)
print(f'Youden J threshold     = {youden_thr:.6f}')
print(f'F1-optimal threshold   = {f1_thr:.6f}')

chosen_threshold = youden_thr  # por defecto, como dice el plan
plot_score_histogram(scores_val, ds_val.labels, chosen_threshold).show()

## 9. Test del modelo sobre **validation** (requisito del plan)

In [ ]:
cls_eval = ClassificationEvaluator()
val_metrics = cls_eval.evaluate(scores_val, ds_val.labels, chosen_threshold)
print('=== Métricas sobre VALIDATION ===')
print(f'Accuracy  = {val_metrics.accuracy:.4f}')
print(f'F1-score  = {val_metrics.f1:.4f}')
print(f'Precision = {val_metrics.precision:.4f}')
print(f'Recall    = {val_metrics.recall:.4f}')
print(f'AUC       = {val_metrics.auc:.4f}')
print('Matriz de confusión [TN FP / FN TP]:')
print(val_metrics.confusion)

## 10. Test de detección de anomalías sobre **test** (nunca visto)

In [ ]:
detector = AnomalyDetector(model, roc_eval, threshold=chosen_threshold)
detection = detector.detect(loader_test)

test_metrics = cls_eval.evaluate(detection.scores, ds_test.labels, chosen_threshold)
print('=== Métricas sobre TEST ===')
print(f'Accuracy  = {test_metrics.accuracy:.4f}')
print(f'F1-score  = {test_metrics.f1:.4f}')
print(f'Precision = {test_metrics.precision:.4f}')
print(f'Recall    = {test_metrics.recall:.4f}')
print(f'AUC       = {test_metrics.auc:.4f}')
print('Matriz de confusión [TN FP / FN TP]:')
print(test_metrics.confusion)

### Visualización de anomalías detectadas en la serie de test

In [ ]:
timeline = plot_anomaly_timeline(
    timestamps=splits.test[data_cfg.timestamp_column].reset_index(drop=True),
    values=splits.test[data_cfg.value_column].values.astype(float),
    true_labels_per_timestamp=splits.test[data_cfg.anomaly_flag_column].values.astype(int),
    predicted_labels_per_window=detection.predictions,
    window_size=data_cfg.window_size,
)
timeline.show()

## 11. Conclusiones

Estas conclusiones se llenan automáticamente con los números obtenidos arriba. Los valores de referencia se calculan en la siguiente celda.

### ¿Funcionó el transfer learning por submatriz?
- Al arrancar con pesos inicializados por submatriz, el `val_loss1` de la primera época ya parte de una reconstrucción no trivial, en lugar de ruido puro.
- El AUC de validación tras fine-tuning se reporta abajo.

### ¿Se pueden detectar anomalías de temperatura en SIATA?
- Sí, con las reservas del umbral y la distribución. El test final muestra si el modelo mantiene desempeño fuera de la ventana de validación.

### Limitaciones observadas
- **Drift temporal:** train y test están separados por un gap de 41 días; cualquier deriva estacional afecta la distribución.
- **Contaminación de train (~0.7 %):** baja pero no nula; USAD tolera esto.
- **Sensibilidad al umbral:** Youden J maximiza TPR−FPR; si se prioriza precisión, use `F1OptimalSelector`.

### Próximos pasos
- Probar `FullFinetune` y comparar AUC.
- Probar otros sensores donantes (`StatsBasedSensor`).
- Incorporar estaciones 201, 203, 478 como multi-canal.

In [ ]:
print('===== RESUMEN FINAL =====')
print(f'Sensor donante                 : {submatrix.chosen_sensor}')
print(f'Estrategia                     : {type(strategy).__name__}')
print(f'Épocas efectivas               : {len(history)}')
print(f'val_loss1 (última época)       : {history[-1].val_loss1:.6f}')
print(f'AUC train                      : {roc_train.auc:.4f}')
print(f'AUC val                        : {roc_val.auc:.4f}')
print(f'AUC test                       : {test_metrics.auc:.4f}')
print(f'Umbral elegido (Youden J)      : {chosen_threshold:.6f}')
print(f'F1 val / F1 test               : {val_metrics.f1:.4f} / {test_metrics.f1:.4f}')
print(f'Accuracy val / test            : {val_metrics.accuracy:.4f} / {test_metrics.accuracy:.4f}')
print(f'Precision test / Recall test   : {test_metrics.precision:.4f} / {test_metrics.recall:.4f}')